# Model Context Protocol

Até aqui toda ferramenta foi uma função Python escrita no mesmo notebook do agente. Para usar a mesma ferramenta em outro programa, como o Claude Desktop, o Cursor ou o agente de outra equipe, seria preciso copiar o código e adaptá-lo ao formato de cada um.

O Model Context Protocol, ou MCP, resolve isso separando as duas pontas. As ferramentas ficam em um programa à parte, o servidor, e qualquer cliente que fale o protocolo pergunta ao servidor quais ferramentas existem e pede que ele execute uma delas. Este notebook escreve um servidor com as ferramentas de viagem da aula de agentes, olha as mensagens que ele troca e liga esse servidor, e depois um servidor de terceiros, a um agente.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langgraph mcp langchain-mcp-adapters
# !pip install -q langchain-groq langchain-google-genai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import json
import subprocess
import sys

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)

## Um servidor MCP

Um servidor MCP é um programa comum. O SDK oficial do MCP para Python traz o `FastMCP`, que transforma funções em ferramentas do protocolo com um decorador, do mesmo jeito que o `@tool` do LangChain: o nome vem da função, a descrição vem da docstring e os parâmetros vêm da assinatura.

O servidor abaixo tem as duas ferramentas do assistente de viagem da aula de agentes. A célula grava o código no arquivo `travel_server.py`, porque o servidor vai rodar como um processo separado do notebook.

In [ ]:
%%writefile travel_server.py
import json
import urllib.request
from urllib.parse import quote

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("viagem", log_level="WARNING")


def fetch(url: str) -> dict:
    """Faz um GET e devolve o JSON da resposta."""
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


@mcp.tool()
def get_weather(city: str) -> str:
    """Devolve a previsão de temperatura e chuva dos próximos três dias em uma cidade."""
    place = fetch(f"https://geocoding-api.open-meteo.com/v1/search?name={quote(city)}&count=1&language=pt")["results"][0]
    daily = fetch(
        f"https://api.open-meteo.com/v1/forecast?latitude={place['latitude']}&longitude={place['longitude']}"
        "&daily=temperature_2m_min,temperature_2m_max,precipitation_probability_max&forecast_days=3&timezone=auto"
    )["daily"]
    return f"{place['name']}, {place['country']}\n" + "\n".join(f"{day}: {low} a {high} °C, chuva {rain}%" for day, low, high, rain in zip(*daily.values()))


@mcp.tool()
def convert_currency(amount: float, source: str, target: str) -> str:
    """Converte um valor entre moedas pelo código de três letras, como BRL, EUR ou USD."""
    rates = fetch(f"https://api.frankfurter.dev/v1/latest?amount={amount}&from={source}&to={target}")["rates"]
    return f"{amount} {source} = {rates[target]} {target}"


# sem argumentos, o servidor conversa pela entrada e pela saída padrão
mcp.run()

O arquivo não faz nada sozinho: quem o executa é o cliente, que inicia o servidor como subprocesso e conversa com ele pela entrada e pela saída padrão. Esse jeito de conectar se chama transporte stdio, e é o mais comum para servidores que rodam na mesma máquina.

## As mensagens

Antes de usar uma biblioteca cliente, vale ver o que passa pelo canal. Cada mensagem é um JSON em uma linha. Um pedido tem um `id`, o nome do método e os parâmetros, e a resposta volta com o mesmo `id`. Esse formato é o JSON-RPC, e o MCP só define quais métodos existem.

A célula abaixo inicia o servidor e cria a função `send`, que escreve uma mensagem na entrada do servidor e lê a resposta na saída.

In [ ]:
server = subprocess.Popen([sys.executable, "travel_server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)


def send(message: dict) -> dict | None:
    """Manda uma mensagem ao servidor e, se ela tiver id, lê a resposta."""
    server.stdin.write(json.dumps(message) + "\n")
    server.stdin.flush()
    if "id" in message:
        return json.loads(server.stdout.readline())

A conversa começa com um aperto de mão. O cliente se apresenta no `initialize`, o servidor responde com o nome dele e o que oferece, e o cliente avisa que terminou. O aviso não tem `id`, então não tem resposta.

In [ ]:
hello = send({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "notebook", "version": "1.0"}},
})
send({"jsonrpc": "2.0", "method": "notifications/initialized"})

print(hello["result"]["serverInfo"])

Com o aperto de mão feito, o cliente pede a lista de ferramentas com o método `tools/list`.

In [ ]:
listed = send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})

for spec in listed["result"]["tools"]:
    print(spec["name"], "|", spec["description"])
    print(spec["inputSchema"])

É o que o modelo precisa saber para usar as ferramentas: nome, descrição e parâmetros em JSON Schema. O `amount` virou `number` porque a anotação é `float`. Por fim, o cliente pede uma execução com o método `tools/call`.

In [ ]:
called = send({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {"name": "convert_currency", "arguments": {"amount": 500, "source": "BRL", "target": "EUR"}},
})

print(called["result"]["content"])

A conversão rodou dentro do servidor, e o notebook só recebeu o texto. Isso é todo o protocolo que um agente usa: listar as ferramentas para montar o prompt, e chamar uma delas quando o modelo pedir. Encerrar o processo encerra o servidor.

In [ ]:
server.terminate()

### Exercício 1

Acrescente ao `travel_server.py` uma ferramenta `get_holidays`, que recebe o ano e o código de duas letras de um país, como PT, e devolve os feriados nacionais, usando a API pública `https://date.nager.at/api/v3/PublicHolidays/{ano}/{pais}`. Grave o arquivo de novo, reinicie o servidor e refaça o aperto de mão, o `tools/list` e um `tools/call` com a ferramenta nova. Que descrição e que parâmetros apareceram para ela?

In [ ]:
# Seu código aqui

## O servidor no agente

Escrever as mensagens à mão serve para ver o protocolo, mas um agente precisa de ferramentas do LangChain. O `MultiServerMCPClient`, do pacote `langchain-mcp-adapters`, faz a ponte: recebe um dicionário com um servidor por chave, inicia cada um, faz o aperto de mão e o `tools/list`, e devolve cada ferramenta como uma ferramenta comum do LangChain.

In [ ]:
client = MultiServerMCPClient(
    {"viagem": {"transport": "stdio", "command": sys.executable, "args": ["travel_server.py"]}}
)

tools = await client.get_tools()

for mcp_tool in tools:
    print(mcp_tool.name, "|", mcp_tool.description)

Daqui em diante é o `create_agent` da aula de agentes. A única diferença é que a chamada é `ainvoke`, com `await`, porque cada execução de ferramenta é uma troca de mensagens com outro processo e as ferramentas do adaptador só funcionam de forma assíncrona. O Jupyter aceita `await` direto na célula.

In [ ]:
travel_agent = create_agent(model, tools=tools, system_prompt="Você é um assistente de viagem. Responda em no máximo quatro frases.")

result = await travel_agent.ainvoke(
    {"messages": [HumanMessage("Vou passar 3 dias em Lisboa com 2000 reais. Como vai estar o tempo e quanto dá isso em euros?")]}
)

for message in result["messages"]:
    print(message.type, ":", message.tool_calls if getattr(message, "tool_calls", None) else message.text)

O agente não sabe que as ferramentas vêm de outro processo. Ele viu dois esquemas, pediu duas chamadas, e o adaptador transformou cada pedido em um `tools/call` como o que foi escrito à mão na seção anterior.

### Exercício 2

O `search_wikipedia` do notebook de RAG agêntico só existe dentro daquele notebook. Escreva um servidor `wiki_server.py` com o `FastMCP` que exponha a mesma busca, acrescente-o como uma segunda chave no dicionário do `MultiServerMCPClient` e monte um agente com as ferramentas dos dois servidores. Peça algo que precise de ambos, como um pouco da história de uma cidade e a previsão do tempo lá. Que ferramentas foram chamadas, e de qual servidor veio cada uma?

In [ ]:
# Seu código aqui

## Um servidor de terceiros

A vantagem do protocolo aparece quando o servidor foi escrito por outra pessoa. Existem servidores MCP prontos para GitHub, bancos de dados, navegadores, arquivos e muitos outros serviços, e alguns rodam em outra máquina. Nesse caso o transporte é HTTP, e no lugar do comando vai o endereço do servidor.

O DeepWiki é um servidor público, sem chave, que responde perguntas sobre repositórios do GitHub.

In [ ]:
remote = MultiServerMCPClient({"deepwiki": {"transport": "streamable_http", "url": "https://mcp.deepwiki.com/mcp"}})

wiki_tools = await remote.get_tools()

for mcp_tool in wiki_tools:
    print(mcp_tool.name, "|", mcp_tool.description[:90])

In [ ]:
wiki_agent = create_agent(model, tools=wiki_tools, system_prompt="Responda em português, em no máximo três frases.")

result = await wiki_agent.ainvoke({"messages": [HumanMessage("No repositório langchain-ai/langchain-mcp-adapters, para que serve o MultiServerMCPClient?")]})

print(result["messages"][1].tool_calls)
print(result["messages"][-1].text)

Ninguém nesta sala escreveu essas ferramentas, e elas entraram no agente com três linhas. Pelo mesmo motivo, a descrição e a saída delas vão para o prompt sem que ninguém as tenha lido, e um servidor mal-intencionado pode colocar instruções ali. Instalar um servidor MCP pede o mesmo cuidado que instalar uma dependência; o assunto volta na aula de guardrails e segurança.

### Exercício 3

Escolha um repositório do GitHub que você usa e faça três perguntas sobre ele ao `wiki_agent`: uma geral, uma sobre um arquivo ou função específica e uma sobre algo que o repositório não tem. Qual ferramenta o agente escolheu em cada caso, e como ele respondeu à pergunta sem resposta?

In [ ]:
# Seu código aqui